In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from tqdm import tqdm
tqdm.pandas()   # supaya bisa pakai progress_apply

In [ ]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Kuliah\Semester 7\Fuzzy Logic\Proyek\daily_sentiment.csv")
df.head()

In [ ]:
df.info()

### RoBERTa

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("w11wo/indonesian-roberta-base-sentiment-classifier")
model = AutoModelForSequenceClassification.from_pretrained("w11wo/indonesian-roberta-base-sentiment-classifier")

In [ ]:
def get_roberta_scores(text):
    if pd.isna(text) or str(text).strip() == "":
        return pd.Series([0.0, 0.0, 0.0, 0.0])

    text = str(text)

    # Batasi panjang (RoBERTa maksimum ~512 token)
    max_length = 500
    tokens = tokenizer.tokenize(text)
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
        text = tokenizer.convert_tokens_to_string(tokens)

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    # Build label->prob dict using model.config.id2label (robust terhadap urutan indeks)
    id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
    prob_by_label = {id2label[i]: float(probs[i]) for i in range(len(probs))}

    # Ambil nilai pasti (default 0.0 jika label tak ditemukan)
    neg = prob_by_label.get('negative', 0.0)
    neu = prob_by_label.get('neutral',  0.0)
    pos = prob_by_label.get('positive', 0.0)

    # Compound-like score: positif - negatif (range [-1, 1])
    score = pos - neg

    return pd.Series([neg, neu, pos, score])


In [ ]:
df[['r_neg', 'r_neu', 'r_pos', 'roberta_compound']] = \
    df['Article'].progress_apply(get_roberta_scores)


In [ ]:
print(df[['Article', 'r_neg', 'r_neu', 'r_pos', 'roberta_compound']].head(10))

In [ ]:
# show how model labels map to indices
print(model.config.id2label)   # misal -> {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}


### Fuzzifikasi

In [ ]:
# === Universe ===
compound = ctrl.Antecedent(np.arange(-1, 1.01, 0.01), 'compound')
sentiment = ctrl.Consequent(np.arange(0, 1.01, 0.01), 'sentiment')

compound['negative'] = fuzz.trimf(compound.universe, [-1, -1, -0.1])
compound['neutral']  = fuzz.trimf(compound.universe, [-0.3, 0, 0.3])
compound['positive'] = fuzz.trimf(compound.universe, [0.1, 1, 1])


compound['slightly_neg'] = fuzz.trimf(compound.universe, [-0.40, -0.20, 0.00])
compound['slightly_pos'] = fuzz.trimf(compound.universe, [0.00, 0.20, 0.40])

# === Membership function for sentiment output ===
sentiment['neg'] = fuzz.trimf(sentiment.universe, [0.00, 0.00, 0.35])
sentiment['neu'] = fuzz.trimf(sentiment.universe, [0.25, 0.50, 0.75])
sentiment['pos'] = fuzz.trimf(sentiment.universe, [0.65, 1.00, 1.00])
# ======================================================
# MEMBERSHIP FUNCTION TABLES
# ======================================================
compound_x = compound.universe
compound_df = pd.DataFrame({
    "compound": compound_x,
    "negative": compound['negative'].mf,
    "neutral": compound['neutral'].mf,
    "positive": compound['positive'].mf,
})

sentiment_x = sentiment.universe
sentiment_df = pd.DataFrame({
    "sentiment": sentiment_x,
    "neg": sentiment['neg'].mf,
    "neu": sentiment['neu'].mf,
    "pos": sentiment['pos'].mf
})

print("=== Compound Membership Table (last 10 rows) ===")
print(compound_df.tail(10))

print("\n=== Sentiment Membership Table (first 10 rows) ===")
print(sentiment_df.head(10))

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(compound_x, compound['negative'].mf, label='negative')
plt.plot(compound_x, compound['neutral'].mf, label='neutral')
plt.plot(compound_x, compound['positive'].mf, label='positive')
plt.title("Compound Membership Functions")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,6))
plt.plot(sentiment_x, sentiment['neg'].mf, label='neg')
plt.plot(sentiment_x, sentiment['neu'].mf, label='neu')
plt.plot(sentiment_x, sentiment['pos'].mf, label='pos')
plt.title("Sentiment Output Membership Functions")
plt.legend()
plt.grid(True)
plt.show()


### Rule Base Fuzzy

In [ ]:
rule1 = ctrl.Rule(compound['negative'], sentiment['neg'])
rule2 = ctrl.Rule(compound['neutral'],  sentiment['neu'])
rule3 = ctrl.Rule(compound['positive'], sentiment['pos'])

### Inferensi

In [ ]:
sentiment_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3,
])

sentiment_sim = ctrl.ControlSystemSimulation(sentiment_ctrl)


### Defuzzifikasi

In [ ]:
predicted_labels = []

for c in df['roberta_compound']:
    sentiment_sim.input['compound'] = float(c)
    sentiment_sim.compute()
    score = sentiment_sim.output['sentiment']

    if score < 0.33:
        predicted_labels.append("negative")
    elif score < 0.6:
        predicted_labels.append("neutral")
    else:
        predicted_labels.append("positive")

df['FuzzyLabel'] = predicted_labels


### Evaluasi

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(df['SentimentLabel'], df['FuzzyLabel']))
print(classification_report(df['SentimentLabel'], df['FuzzyLabel']))

In [ ]:
df.info()

In [ ]:
df[['Article', 'SentimentLabel', 'FuzzyLabel']].head(20)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Buat figure dengan 2 subplot sejajar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot distribusi label asli ---
sns.countplot(x=df['SentimentLabel'], ax=axes[0])
axes[0].set_title("Distribusi Label Asli")
axes[0].set_xlabel("Label Asli")
axes[0].set_ylabel("Jumlah")

# --- Plot distribusi label fuzzy (prediksi) ---
sns.countplot(x=df['FuzzyLabel'], ax=axes[1])
axes[1].set_title("Distribusi Label Fuzzy (Prediksi)")
axes[1].set_xlabel("Label Prediksi")
axes[1].set_ylabel("Jumlah")

plt.tight_layout()
plt.show()
